# Per-cell optogenetic dosing via an inference server

This demo shows faro's closed loop for **per-cell** stimulation dosing:

1. a frame is acquired and cells are segmented + tracked (stable `particle` IDs);
2. per-cell features are sent to an **inference server**, which returns a
   **stim exposure (ms) per cell**;
3. faro delivers those distinct doses on a binary DMD via a **cumulative
   "staircase"**: all cells start lit, and each cell's mirrors switch off once
   it reaches its target dose. Total light-delivery time = the *maximum*
   exposure, not the sum.

Here the server is a `FakeInferenceClient` (in-process, no network), so the
whole loop runs with no cluster and no trained model. Swapping in the real
server is a one-line change (last cell).

**Part A** exercises the real `InferenceServerStim` + `FE_ErkKtr` path on a
synthetic ERK-KTR frame. **Part B** runs the full end-to-end controller loop.

In [ ]:
import sys, os
# allow importing the repo when run from anywhere
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# This notebook needs faro's environment. If the import below fails, your
# Jupyter *kernel* isn't faro's venv (activating the venv in a shell does not
# change a running kernel). Fix: Kernel -> Change Kernel -> 'Python (faro)'.
try:
    import useq  # a faro core dependency
except ModuleNotFoundError as e:
    raise RuntimeError(
        f"faro's dependencies aren't importable in this kernel ({e}). "
        "Select the 'Python (faro)' kernel (Kernel -> Change Kernel), or run "
        "`uv run python -m ipykernel install --user --name faro-venv "
        "--display-name 'Python (faro)'` from the faro repo and pick it."
    ) from e

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

## Part A — one frame, end to end

### A synthetic ERK-KTR frame with a gradient of activity

Channel 0 is the nuclei (for segmentation); channel 1 is the biosensor, with a
cytosolic ring intensity set so each cell has a known cytosol/nucleus ratio
(`cnr`) increasing across the grid.

In [ ]:
def make_erk_frame(n_side=3, img_size=384, R=14, cnr_lo=0.5, cnr_hi=2.5, b_nuc=8000):
    c0 = np.zeros((img_size, img_size), np.uint16)
    c1 = np.zeros((img_size, img_size), np.uint16)
    yy, xx = np.ogrid[:img_size, :img_size]
    step = img_size // (n_side + 1)
    cnr_targets = np.linspace(cnr_lo, cnr_hi, n_side * n_side)
    k = 0
    for i in range(n_side):
        for j in range(n_side):
            cy, cx = step * (i + 1), step * (j + 1)
            rr = (yy - cy) ** 2 + (xx - cx) ** 2
            nucleus = rr <= R ** 2
            ring = (rr > R ** 2) & (rr <= (R + 5) ** 2)
            c0[nucleus] = 40000
            c1[nucleus] = b_nuc
            c1[ring] = int(cnr_targets[k] * b_nuc)
            k += 1
    return np.stack([c0, c1], axis=0), cnr_targets  # img is [C, H, W]

img, cnr_targets = make_erk_frame()

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
ax[0].imshow(img[0], cmap='gray'); ax[0].set_title('C0 — nuclei'); ax[0].axis('off')
ax[1].imshow(img[1], cmap='magma'); ax[1].set_title('C1 — ERK-KTR biosensor'); ax[1].axis('off')
plt.tight_layout(); plt.show()

### Segment + extract per-cell features (`FE_ErkKtr`)

The feature extractor recovers each cell's `cnr` / `cnr_median` — exactly the
columns the real model consumes.

In [ ]:
from faro.segmentation.base import OtsuSegmentator
from faro.feature_extraction.erk_ktr import FE_ErkKtr

labels = OtsuSegmentator().segment(img[0])
fe = FE_ErkKtr('labels')
feats, _ = fe.extract_features({'labels': labels}, img, None, {})
feats[['label', 'cnr', 'cnr_median', 'area_nuc']].round(3)

### Ask the inference server for a per-cell dose

`InferenceServerStim` computes the features in-hook, sends them to the client,
and turns the returned per-cell exposures into the staircase. Here the fake
client maps `cnr` → exposure (ms), clipped to the rig's 0–3000 ms band.

In [ ]:
from faro.inference.client import FakeInferenceClient
from faro.stimulation.inference_server import InferenceServerStim

# stand-in for the model: higher ERK activity -> longer dose
client = FakeInferenceClient(rule=lambda c: float(np.clip(c['cnr'] * 1000.0, 0, 3000)))

# tracks carries the stable per-cell `particle` id (here 1:1 with label)
tracks = pd.DataFrame({'label': feats['label'].values})
tracks['particle'] = tracks['label']
tracks['fname'] = '000_00000'

stim = InferenceServerStim(client, feature_extractor=fe, eps_ms=25.0)
subframes, _ = stim.get_stim_mask(
    {'labels': labels},
    metadata={'fov': 0, 'timestep': 0, 'fname': '000_00000'},
    img=img, tracks=tracks,
)
durations = [d for _, d in subframes]
print(f'{len(subframes)} DMD sub-frames')
print('per-sub-frame LED time (ms):', [round(d, 1) for d in durations])
print('total light-delivery time (ms):', round(sum(durations), 1))
print('max single-cell exposure   (ms):', round(max(np.clip(cnr_targets*1000, 0, 3000)), 1))

### The staircase

Each sub-frame lights every cell that still needs dose; cells drop out as they
reach their target. The mask shrinks step by step — and the **total time equals
the longest single exposure**, independent of cell count.

In [ ]:
n = len(subframes)
cols = min(n, 5)
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3*rows))
axes = np.atleast_1d(axes).ravel()
cum = 0.0
for i, (mask, dur) in enumerate(subframes):
    cum += dur
    axes[i].imshow(mask, cmap='gray'); axes[i].axis('off')
    axes[i].set_title(f'step {i}\n+{dur:.0f} ms (t={cum:.0f})', fontsize=9)
for j in range(n, len(axes)):
    axes[j].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# each cell's delivered on-time should equal its requested exposure
requested = np.clip(cnr_targets * 1000.0, 0, 3000)
requested = np.round(requested / 25.0) * 25.0   # same 25 ms grid the staircase uses
fig, ax = plt.subplots(figsize=(6, 4))
order = np.argsort(requested)
ax.bar(range(len(requested)), np.sort(requested), color='#4C78A8')
ax.set_xlabel('cell (sorted by dose)'); ax.set_ylabel('delivered LED time (ms)')
ax.set_title('Per-cell dose delivered by the staircase')
plt.tight_layout(); plt.show()

## Part B — the full closed loop through the Controller

Same machinery, driven end-to-end on a headless `FakeMicroscope` (no hardware):
acquire → segment → track → `InferenceServerStim` → DMD sub-frames fire. We
record every SLM image the controller dispatches. Two cells of different size
get two different doses, so each stim timepoint fires a 2-step staircase.

*(This cell reuses faro's test harness for a zero-config synthetic scene.)*

In [ ]:
from faro.core.controller import Controller
from faro.feature_extraction.simple import SimpleFE
from faro.tracking.trackpy import TrackerTrackpy
from tests.fake_microscope import FakeMicroscope
from tests.fixtures import CircleScene, make_events, make_pipeline, run_and_wait
import tempfile

class RecScene(CircleScene):
    def __init__(self):
        super().__init__(with_slm=True)
        self.records = []  # (timepoint, on_pixels, exposure_ms)
    def on_slm_displayed(self, image, event):
        super().on_slm_displayed(image, event)
        self.records.append((event.index.get('t', 0),
                             int((np.asarray(image) > 0).sum()), event.exposure))

client_b = FakeInferenceClient(rule=lambda c: 100.0 if c['area'] > 900 else 250.0)
stim_b = InferenceServerStim(client_b, feature_extractor=SimpleFE('labels'), eps_ms=25.0)
pipe = make_pipeline(tempfile.mkdtemp(), tracker=TrackerTrackpy(search_range=50, memory=3),
                     stimulator=stim_b)
scene = RecScene()
ctrl = Controller(FakeMicroscope(scene), pipe)
run_and_wait(ctrl, make_events(5, stim_frames=(2, 3)), stim_mode='current')

print('background errors:', ctrl.background_errors)
pd.DataFrame(scene.records, columns=['timepoint', 'on_pixels', 'exposure_ms'])

## Pointing at the real inference server

Everything above is transport-agnostic. To drive dosing from the real
cluster model, swap the client — nothing else changes:

```python
from faro.inference.client import HttpInferenceClient
client = HttpInferenceClient('http://localhost:8080')  # e.g. an SSH tunnel to the GPU node
stim = InferenceServerStim(client, feature_extractor=FE_ErkKtr('labels'), eps_ms=25.0)
```

The server contract (request/response schema, per-`(fov, particle)` state,
retries) is documented in the inference-server spec. For the real ERK-KTR
model, keep `FE_ErkKtr` so the payload already carries `cnr` / `cnr_median`.